In [1]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

In [2]:
PROJECT_ROOT=Path.cwd().parent
# Khai báo đường dẫn
DATA_DIR=PROJECT_ROOT/"data"/"raw"

PARQUET_PATH=DATA_DIR/"yellow_tripdata_2026-01.parquet"
LOOKUP_PATH=DATA_DIR/"taxi_zone_lookup.csv"

# Kiểm tra file tồn tại
print(DATA_DIR)
print("Project root: ",PROJECT_ROOT)
print("Parquet exists: ",PARQUET_PATH.exists())
print("Lookup exists: ",LOOKUP_PATH.exists())

c:\Users\Acer\Documents\DATA ANALYST\Portfolio\NYC Taxi Project\data\raw
Project root:  c:\Users\Acer\Documents\DATA ANALYST\Portfolio\NYC Taxi Project
Parquet exists:  True
Lookup exists:  True


In [3]:
df=pd.read_parquet(PARQUET_PATH)
zones=pd.read_csv(LOOKUP_PATH)

display(df.head(10))
display(zones.head(10))

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75
7,1,2026-01-01 00:34:28,2026-01-01 00:59:05,0.0,2.90,1.0,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.00,2.5,0.0,0.75
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
9,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3.0,1.83,1.0,N,237,263,1,10.7,1.00,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [4]:
# Tạo bản sao để xử lý
df_clean=df.copy()
zones_clean=zones.copy()
print("Dữ liệu df ban đầu: ",df.shape)
print("Dữ liệu zones copy để làm sạch: ",df_clean.shape)
print("Dữ liệu zones ban đầu: ",zones.shape)
print("Dữ liệu zones copy để làm sạch: ",zones_clean.shape)

Dữ liệu df ban đầu:  (3724889, 20)
Dữ liệu zones copy để làm sạch:  (3724889, 20)
Dữ liệu zones ban đầu:  (265, 4)
Dữ liệu zones copy để làm sạch:  (265, 4)


In [5]:
# Thêm cột tính tổng thời gian mỗi chuyên đi để cho dễ dàng phân tích thời gian trung bình chuyến đi
df_clean['trip_duration_minutes']=(
    df_clean['tpep_dropoff_datetime']
    -df_clean['tpep_pickup_datetime']
).dt.total_seconds()/60

In [6]:
df_clean

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_minutes
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,5.550000
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,5.716667
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75,8.883333
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,42.800000
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75,13.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,None,237,161,0,...,0.00,0.5,0.00,0.0,1.0,21.84,NaN,NaN,0.75,13.266667
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,None,42,42,0,...,0.00,0.5,0.00,0.0,1.0,24.69,NaN,NaN,0.00,0.233333
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,None,137,69,0,...,0.00,0.5,0.00,0.0,1.0,33.96,NaN,NaN,0.75,15.783333
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,None,137,162,0,...,0.00,0.5,0.00,0.0,1.0,23.94,NaN,NaN,0.75,9.650000


In [7]:
numeric_cols=df.select_dtypes(include='number').columns
numeric_cols

Index(['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [8]:
checkIssuesCol=pd.DataFrame({
    'ZeroCols':df_clean[numeric_cols].eq(0).sum(),
    'Negativecols':df_clean[numeric_cols].lt(0).sum()
})

checkIssuesCol

,ZeroCols,Negativecols
VendorID,0,0
passenger_count,14787,0
trip_distance,125738,0
RatecodeID,0,0
PULocationID,0,0
DOLocationID,0,0
payment_type,1088058,0
fare_amount,2082,39463
extra,2270914,20035
mta_tax,48423,37730


### Xóa dòng total amount = 0 với payment type =(1,2)

In [ ]:
df_TA_PA12=df_clean[(df['total_amount']==0) & ((df['payment_type']==1) | (df['payment_type']==2))]
df_clean=df_clean.drop(df_TA_PA12.index)


C:\Users\Acer\AppData\Local\Temp\ipykernel_68136\1645147905.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_TA_PA12=df_clean[(df['total_amount']==0) & ((df['payment_type']==1) | (df['payment_type']==2))]


- Xóa dòng âm không ghép được dòng dương

In [27]:
cols = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
        'PULocationID', 'DOLocationID']

taNegative=df_clean[df_clean['total_amount']<0].copy()
taPositive=df_clean[df_clean['total_amount']>0].copy()

# Chuyên đôi cột giá trị âm thành dương để lọc ra có cặp
taNegative['tong_tien']=-taNegative['total_amount']
taPositive['tong_tien']=taPositive['total_amount']

taNegative=taNegative.reset_index()
taPositive['co_cap']=True

results= taNegative.merge(taPositive,on=cols+['tong_tien'],how='left')
results['co_cap']=results['co_cap'].eq(True)

rowDelete=results[results['co_cap']==False]['index']
n = len(df_clean)
df_clean = df_clean.drop(rowDelete)
print("Đã xóa dòng âm không ghép được:", n - len(df_clean))

Đã xóa dòng âm không ghép được: 4277


- Xóa tốc độ lớn hơn 80mph


In [29]:
# v=s/t
gio=df_clean['trip_duration_minutes']/60
speed=df_clean['trip_distance']/gio.where(gio>0)
n = len(df_clean)
df_clean = df_clean[~(speed > 80)].copy()
print("Đã xóa tốc độ > 80 mph:", n - len(df_clean))

Đã xóa tốc độ > 80 mph: 873


In [30]:
print('Data trước khi xóa', df.shape)
print('Data sau khi xoa: ',df_clean.shape)

Data trước khi xóa (3724889, 20)
Data sau khi xoa:  (3719659, 21)


In [32]:
display(zones_clean)

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN


### Kết luận bảng zones (taxi_zone_lookup)

- Bảng có 265 dòng, mỗi dòng là một khu vực (LocationID), không có dòng trùng lặp.
- Có 2 dòng chứa giá trị NULL:
  - LocationID 264: Borough = Unknown, Zone và service_zone NULL.
  - LocationID 265: Zone =Outside of NYC, Borough và service_zone NULL.
  - Đây là hai mã đặc biệt của TLC (khu vực không xác định và ngoài NYC), không phải lỗi dữ liệu.

### Xuất ra file đã làm sạch

In [33]:
OUT_DIR=PROJECT_ROOT/"data"/"processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(OUT_DIR/"trip_clean.parquet",index=False)
zones.to_csv(OUT_DIR/"taxi_zones.csv",index=False)
print('Kích thước cuối: ',df_clean.shape)

Kích thước cuối:  (3719659, 21)
